# Esplorativa Altre Fonti Dati

Notebook per testare fonti dati alternative a BDAP

## Fonti da Esplorare

1. **OpenBilanci** (Openpolis)
   - Bilanci armonizzati 2008-2023
   - Formato user-friendly
   - PROBLEMA: Certificato SSL scaduto

2. **SIOPE+** (MEF)
   - Sistema evoluto SIOPE
   - Dati real-time
   - PROBLEMA: Richiede autenticazione

3. **Dati.gov.it**
   - Portale Open Data nazionale
   - Aggregatore dataset pubblici

4. **Ministero dell'Interno**
   - Certificati bilancio consuntivi
   - PROBLEMA: Richiede navigazione web/scraping

**NOTA:** Tutto il codice in questo notebook e' esplorativo/bozza.
Da sviluppare solo se BDAP non e' sufficiente.

---

## Setup

In [ ]:
import requests
import pandas as pd
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Setup cartelle
data_raw = Path('../data/raw')
data_raw.mkdir(parents=True, exist_ok=True)
(data_raw / 'openbilanci').mkdir(exist_ok=True)
(data_raw / 'datigov').mkdir(exist_ok=True)
(data_raw / 'mininterno').mkdir(exist_ok=True)

print("Setup completato")

---

# 1. OpenBilanci (Openpolis)

**URL:** https://openbilanci.it

**Vantaggi:**
- Bilanci armonizzati 15 anni
- Interfaccia user-friendly
- Confronti automatici

**Problemi:**
- Certificato SSL scaduto (al momento test)
- Nessuna API pubblica documentata
- Richiede web scraping

**Stato:** Da testare quando certificato SSL sara' rinnovato

In [ ]:
def test_openbilanci():
    """
    Test connessione OpenBilanci
    """
    
    base_url = "https://openbilanci.it"
    
    print("Test OpenBilanci")
    print("="*70)
    
    try:
        # Test homepage (disabilita verifica SSL per test)
        response = requests.get(base_url, verify=False, timeout=10)
        
        if response.status_code == 200:
            print("Homepage raggiungibile")
            print(f"Content-Length: {len(response.content)} bytes")
            
            # TODO: Analizzare HTML per capire struttura
            # TODO: Identificare endpoint API se esistenti
            # TODO: Testare scarico dati specifici
            
        else:
            print(f"Status code: {response.status_code}")
    
    except Exception as e:
        print(f"ERRORE: {e}")
    
    print("\nCONCLUSIONE: Richiede approfondimento")
    print("Opzioni:")
    print("  1. Attendere fix certificato SSL")
    print("  2. Reverse engineering API (analisi network)")
    print("  3. Web scraping con Selenium")


# Disabilita warning SSL
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Decommentare per testare:
# test_openbilanci()

---

# 2. SIOPE+ (MEF)

**URL:** https://www.siope.it/Siope2Web/

**Vantaggi:**
- Dati real-time
- Piu dettagliati di SIOPE standard
- Sistema ufficiale MEF

**Problemi:**
- Richiede autenticazione (credenziali PA)
- Non accessibile pubblicamente

**Conclusione:** I dati SIOPE+ finiscono comunque in BDAP dopo qualche mese.
Non necessario per progetto.

In [ ]:
def test_siope_plus():
    """
    Test connessione SIOPE+
    """
    
    base_url = "https://www.siope.it"
    
    print("Test SIOPE+")
    print("="*70)
    
    try:
        response = requests.get(base_url, timeout=10)
        
        if response.status_code == 200:
            print("Portale raggiungibile")
            
            # Verifica se richiede login
            if 'login' in response.text.lower() or 'auth' in response.text.lower():
                print("\nRichiede autenticazione (come previsto)")
            
        else:
            print(f"Status: {response.status_code}")
    
    except Exception as e:
        print(f"ERRORE: {e}")
    
    print("\nCONCLUSIONE: Non accessibile senza credenziali PA")
    print("Alternativa: Usa BDAP per dati SIOPE standard (gia' implementato)")


# Decommentare per testare:
# test_siope_plus()

---

# 3. Dati.gov.it

**URL:** https://www.dati.gov.it

**Vantaggi:**
- Aggregatore nazionale
- API CKAN (come BDAP)
- Dataset da varie fonti

**Svantaggi:**
- Qualita dataset variabile
- Non sempre aggiornati
- Duplicati con BDAP

**Utilizzo suggerito:** Fonte secondaria per dataset specifici non in BDAP

In [ ]:
def test_datigov():
    """
    Test connessione Dati.gov.it
    """
    
    base_url = "https://www.dati.gov.it/api/3/action"
    
    print("Test Dati.gov.it")
    print("="*70)
    
    try:
        # Cerca dataset bilanci comunali
        params = {
            'q': 'bilanci comuni',
            'rows': 20
        }
        
        response = requests.get(
            f"{base_url}/package_search",
            params=params,
            timeout=10
        )
        
        if response.status_code == 200:
            data = response.json()
            
            if data.get('success'):
                count = data['result']['count']
                results = data['result']['results']
                
                print(f"Connessione OK")
                print(f"Dataset 'bilanci comuni': {count}")
                print("\nTop 5 dataset:")
                
                for i, ds in enumerate(results[:5], 1):
                    print(f"\n{i}. {ds.get('title')}")
                    print(f"   Organizzazione: {ds.get('organization', {}).get('title', 'N/A')}")
                    print(f"   Risorse: {len(ds.get('resources', []))}")
                
                # Salva per analisi futura
                output = data_raw / 'datigov' / 'bilanci_comuni_datasets.json'
                with open(output, 'w', encoding='utf-8') as f:
                    json.dump(results, f, indent=2, ensure_ascii=False)
                print(f"\nSalvato: {output}")
                
                return results
        
        print(f"ERRORE: Status {response.status_code}")
        return []
        
    except Exception as e:
        print(f"ERRORE: {e}")
        return []


# Decommentare per testare:
# datigov_results = test_datigov()

---

# 4. Ministero dell'Interno

**URL:** https://finanzalocale.interno.gov.it

**Vantaggi:**
- Fonte ufficiale
- Certificati consuntivi completi
- Dati tutti i comuni

**Problemi:**
- Nessuna API pubblica
- Richiede navigazione web complessa
- Necessario Selenium per automazione

**Stato:** Implementabile con Selenium se necessario.
BDAP potrebbe gia' contenere questi dati.

In [ ]:
def test_ministero_interno():
    """
    Test connessione Ministero Interno
    """
    
    base_url = "https://finanzalocale.interno.gov.it"
    
    print("Test Ministero Interno")
    print("="*70)
    
    try:
        response = requests.get(base_url, timeout=10)
        
        if response.status_code == 200:
            print("Portale raggiungibile")
            print("\nIl portale richiede navigazione web interattiva")
            print("\nDati disponibili:")
            print("  - Certificati consuntivi")
            print("  - Certificati preventivi")
            print("  - Situazione patrimoniale")
            
        else:
            print(f"Status: {response.status_code}")
    
    except Exception as e:
        print(f"ERRORE: {e}")
    
    print("\nCONCLUSIONE: Richiede web scraping")
    print("\nImplementazione possibile con:")
    print("  - Selenium (automazione browser)")
    print("  - BeautifulSoup (parsing HTML)")
    print("  - Requests + analisi network per API nascoste")
    print("\nPriorita: Bassa (verificare prima disponibilita in BDAP)")


# Decommentare per testare:
# test_ministero_interno()

---

# 5. Altre Fonti Potenziali

Altre fonti da esplorare in futuro:

**ISTAT:**
- Demografia comuni
- Indicatori socioeconomici
- API: http://dati.istat.it

**Agenzia Entrate:**
- Dichiarazioni fiscali aggregate
- Redditi medi per comune
- Stato: Da verificare disponibilita pubblica

**ANCI (Associazione Comuni Italiani):**
- Studi e ricerche
- Benchmarking comuni
- Stato: Prevalentemente report PDF

**Opencivitas:**
- Dati civici vari
- Community-driven
- Stato: Copertura parziale

In [ ]:
# Template per testare nuove fonti

def test_generic_source(base_url, source_name):
    """
    Template test generico per nuove fonti
    
    Args:
        base_url: URL base fonte
        source_name: Nome fonte
    """
    
    print(f"Test {source_name}")
    print("="*70)
    
    try:
        response = requests.get(base_url, timeout=10)
        
        if response.status_code == 200:
            print("Connessione OK")
            print(f"Content-Type: {response.headers.get('Content-Type')}")
            print(f"Content-Length: {len(response.content)} bytes")
            
            # Analisi base
            content = response.text.lower()
            
            checks = {
                'API disponibile': 'api' in content,
                'Documentazione': 'doc' in content or 'documentation' in content,
                'Download dati': 'download' in content or 'export' in content,
                'Autenticazione richiesta': 'login' in content or 'auth' in content
            }
            
            print("\nAnalisi preliminare:")
            for check, result in checks.items():
                status = "SI" if result else "NO"
                print(f"  {check}: {status}")
            
        else:
            print(f"Status: {response.status_code}")
    
    except Exception as e:
        print(f"ERRORE: {e}")


# Esempi utilizzo:
# test_generic_source("http://dati.istat.it", "ISTAT")
# test_generic_source("https://www.opencivitas.it", "Opencivitas")

---

# Conclusioni e Priorita

## Valutazione Fonti Alternative

**Priorita ALTA (implementare se necessario):**
1. **Dati.gov.it** - API funzionante, facile integrazione
   - Uso: Dataset supplementari non in BDAP
   - Effort: Basso

**Priorita MEDIA (valutare necessita):**
2. **OpenBilanci** - Dati interessanti ma problemi tecnici
   - Uso: Confronti storici, visualizzazioni
   - Effort: Medio (attesa fix SSL o web scraping)

3. **ISTAT** - Demografia e contesto socioeconomico
   - Uso: Arricchimento analisi
   - Effort: Basso-Medio

**Priorita BASSA (solo se strettamente necessario):**
4. **Ministero Interno** - Dati probabilmente gia in BDAP
   - Uso: Certificati consuntivi dettagliati
   - Effort: Alto (Selenium)

5. **SIOPE+** - Non accessibile pubblicamente
   - Uso: Nessuno (dati finiscono in BDAP)
   - Effort: Impossibile senza credenziali

---

## Prossimi Step

1. **Completare analisi BDAP** (notebook 01)
   - Verificare se BDAP contiene gia dati sufficienti
   - Esplorare dataset non-SIOPE in BDAP

2. **Solo se necessario:**
   - Implementare integrazione Dati.gov.it
   - Valutare OpenBilanci quando SSL sara risolto
   - Aggiungere dati ISTAT per contesto

3. **Infrastruttura:**
   - Database unificato
   - ETL per fonti multiple
   - API query layer

---